# Booster / ads extract — Analyses 3 / 4 / 5 / 6

Pulls per-day per-advertiser ads performance to feed:

- **Analysis 3 (Booster Optimisation)** — keyword-targeted Product Listing
- **Analysis 4 (Category Targeting Optimisation)** — category-targeted Product Listing
- **Analysis 5 (Recommendation Asset Analysis)** — Product Recommendation by asset type
- **Analysis 6 (Penetration)** — derived in the app from 3 + 4 + 5

Source: `adserver_analytics.campaign_daily_data` + `lake_adserver.{campaign,advertiser,sub_campaign}` + `lake_cms.gr_category`.

**Backfill structure** — weekly batches, checkpoint-resumable (`_DONE` per batch).
Run cell 4 to extract, cell 6 to merge into the cockpit's `final/` directory.

**Gotchas baked in** (CLAUDE.md §7):
- `cdd.sub_campaign_id <> -1` (the `-1` rows are aggregate roll-ups that double-count)
- 30-day `insert_ds_ist` lookback (Hudi events land up to 30 days late)
- Per-day query — dodges the Trino 20-min cap
- Output already at brand-aggregate grain (advertiser ≈ brand for ads)

In [ ]:
START_DATE = '2026-04-01'
END_DATE   = '2026-06-30'   # bootstrap window. For 90-day backfill: set END=yesterday and START=END-89.

OUT_DIR      = '/home/jovyan'
FINAL_DIR    = '/home/jovyan/final'
BACKFILL_DIR = '/home/jovyan/backfill_booster'

import os, time
from datetime import date, timedelta
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pencilbox as pb

connection = pb.get_connection('[Warehouse] Trino')

def sql(query):
    t0 = time.time()
    df = pd.read_sql_query(sql=query, con=connection)
    print(f'  ok {time.time()-t0:6.1f}s  {len(df):>10,} rows')
    return df

def run_to_parquet_chunked(sql_template, out_path, start, end, batch_size=50_000):
    t0 = time.time()
    writer, schema, total = None, None, 0
    cur_day = date.fromisoformat(start)
    last_day = date.fromisoformat(end)
    while cur_day <= last_day:
        sql_day = sql_template.format(DAY=cur_day.isoformat())
        print(f'  {cur_day.isoformat()}  executing…', flush=True)
        raw = connection.raw_connection()
        cur = raw.cursor()
        td0 = time.time()
        cur.execute(sql_day)
        cols = [d[0] for d in cur.description]
        day_rows = 0
        while True:
            batch = cur.fetchmany(batch_size)
            if not batch: break
            df = pd.DataFrame(batch, columns=cols)
            table = pa.Table.from_pandas(df, preserve_index=False)
            if writer is None:
                schema = table.schema
                writer = pq.ParquetWriter(out_path, schema, compression='snappy')
            else:
                table = table.cast(schema)
            writer.write_table(table)
            day_rows += len(batch)
            print(f'    {time.time()-td0:6.1f}s  {day_rows:>10,} rows', flush=True)
        cur.close(); raw.close()
        total += day_rows
        print(f'  {cur_day.isoformat()}  done {time.time()-td0:6.1f}s  {day_rows:,} rows  (cum {total:,})', flush=True)
        cur_day += timedelta(days=1)
    if writer is not None:
        writer.close()
    size_mb = os.path.getsize(out_path) / 1024**2 if os.path.exists(out_path) else 0
    print(f'  done {out_path}: {total:,} rows, {size_mb:.1f} MB in {time.time()-t0:.0f}s')
    return total

os.makedirs(FINAL_DIR, exist_ok=True)
os.makedirs(BACKFILL_DIR, exist_ok=True)
print(f'Window: {START_DATE} → {END_DATE}')

In [ ]:
# --- Probe: which campaign_type literals exist in the fact table? ---
# Run this once before the first backfill. Look for the Banner Listing literal
# in the output and confirm (or update) CAMPAIGN_TYPE_BANNER in cell 3.
#
# Expected: 'PRODUCT_LISTING' (booster + category), 'PRODUCT_RECOMMENDATION'
# (reco assets), plus the banner one we're after.
probe_sql = f"""
SELECT cdd.campaign_type, COUNT(*) AS rows, SUM(cdd.budget_consumed) AS spend
FROM adserver_analytics.campaign_daily_data cdd
WHERE CAST(from_unixtime(cdd.date_ist) AS DATE) BETWEEN DATE '{START_DATE}' AND DATE '{END_DATE}'
  AND CAST(insert_ds_ist AS DATE) >= DATE '{START_DATE}' - INTERVAL '30' DAY
GROUP BY 1
ORDER BY 3 DESC
"""
sql(probe_sql)

In [ ]:
# --- Probe: how to bridge campaign_id → L0/L1/L2/ptype? ---
# Uses IPython display() so every block's DataFrame actually renders inline
# (sql() returns a DataFrame but only the last expression per cell auto-shows).

from IPython.display import display
import pandas as pd
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 60)

print("=" * 80)
print("A) lake_adserver.sub_campaign — schema")
print("=" * 80)
display(pd.read_sql_query("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'lake_adserver'
  AND table_name   = 'sub_campaign'
ORDER BY ordinal_position
""", connection))

print()
print("=" * 80)
print("B) adserver_analytics.campaign_daily_data — schema (looking for product_id/category)")
print("=" * 80)
display(pd.read_sql_query("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'adserver_analytics'
  AND table_name   = 'campaign_daily_data'
  AND (column_name LIKE '%%product%%'
       OR column_name LIKE '%%category%%'
       OR column_name LIKE '%%ptype%%'
       OR column_name LIKE '%%l0%%'
       OR column_name LIKE '%%l1%%'
       OR column_name LIKE '%%l2%%')
ORDER BY 1
""", connection))

print()
print("=" * 80)
print("C) lake_adserver tables — anything that looks like a campaign↔product bridge?")
print("=" * 80)
display(pd.read_sql_query("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'lake_adserver'
  AND (table_name LIKE '%%product%%'
       OR table_name LIKE '%%category%%'
       OR table_name LIKE '%%campaign_product%%'
       OR table_name LIKE '%%sub_campaign_product%%')
ORDER BY 1
""", connection))

print()
print("=" * 80)
print("D) sub_campaign sample — 5 raw rows (no FK join since we don't know its column names yet)")
print("=" * 80)
display(pd.read_sql_query("""
SELECT * FROM lake_adserver.sub_campaign LIMIT 5
""", connection))

print()
print("=" * 80)
print("E) lake_adserver.campaign — schema")
print("=" * 80)
display(pd.read_sql_query("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'lake_adserver'
  AND table_name   = 'campaign'
ORDER BY ordinal_position
""", connection))


In [ ]:
# --- Probe #2: is lake_adserver.category_keyword_mapping usable? ---
# We need: keyword → (L0, L1, L2, ptype) with the SAME taxonomy the cockpit uses
# (dwh.dim_product). If lake_adserver.category maps 1:1 with that, we skip
# search_etls entirely and do a lightweight JOIN.

from IPython.display import display
import pandas as pd
pd.set_option('display.max_colwidth', 80)

print("A) lake_adserver.category_keyword_mapping — schema")
display(pd.read_sql_query("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'lake_adserver'
  AND table_name   = 'category_keyword_mapping'
ORDER BY ordinal_position
""", connection))

print("\nB) lake_adserver.category — schema (see the hierarchy structure)")
display(pd.read_sql_query("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'lake_adserver'
  AND table_name   = 'category'
ORDER BY ordinal_position
""", connection))

print("\nC) category sample — 10 rows to see the taxonomy shape")
display(pd.read_sql_query("""
SELECT * FROM lake_adserver.category LIMIT 10
""", connection))

print("\nD) category_keyword_mapping sample — 10 rows for common keywords")
display(pd.read_sql_query("""
SELECT * FROM lake_adserver.category_keyword_mapping LIMIT 10
""", connection))

print("\nE) End-to-end test: for keyword 'shampoo', what does the ad-server taxonomy say?")
display(pd.read_sql_query("""
SELECT ckm.*, c.*
FROM lake_adserver.category_keyword_mapping ckm
LEFT JOIN lake_adserver.category c
  ON c.id = ckm.category_id            -- if this errors, we'll fix the FK name
WHERE lower(cast(ckm as varchar)) LIKE '%%shampoo%%'
LIMIT 10
""", connection))


In [ ]:
# --- Probe #3: verify lake_adserver.category L2 names hoist cleanly to dwh.dim_product ---
# If distinct_l1_matches is mostly 1, the L2-name join is clean and we can add
# L0/L1 alongside L2 in the campaign filter dropdowns.

from IPython.display import display
import pandas as pd
pd.set_option('display.max_rows', 40)

print("A) Does lake_adserver.category.category_name align 1:1 with dwh.dim_product.l2_category?")
display(pd.read_sql_query("""
SELECT
    c.category_name                       AS ad_l2,
    count(DISTINCT dp.l1_category)        AS distinct_l1_matches,
    count(DISTINCT dp.l0_category)        AS distinct_l0_matches,
    arbitrary(dp.l0_category)             AS example_l0,
    arbitrary(dp.l1_category)             AS example_l1,
    count(DISTINCT dp.product_id)         AS matching_products
FROM lake_adserver.category c
LEFT JOIN dwh.dim_product dp
       ON dp.l2_category = c.category_name AND dp.is_current = TRUE
WHERE c.category_type = 'L2'
GROUP BY 1
ORDER BY 6 DESC
LIMIT 30
""", connection))

print("\nB) Sanity — pull the ad-server L2 for a real HUL/Loreal keyword ('shampoo')")
display(pd.read_sql_query("""
SELECT
    ckm.keyword,
    c.category_name AS ad_l2,
    dp.l0_category, dp.l1_category, dp.l2_category
FROM lake_adserver.category_keyword_mapping ckm
JOIN lake_adserver.category c ON c.id = ckm.category_id
LEFT JOIN dwh.dim_product dp
       ON dp.l2_category = c.category_name AND dp.is_current = TRUE
WHERE ckm.keyword IN ('shampoo', 'loreal shampoo', 'anti dandruff shampoo', 'face wash', 'sham')
GROUP BY 1, 2, 3, 4, 5
LIMIT 20
""", connection))

print("\nC) Coverage — of all keywords in our booster cube, how many have a category mapping?")
display(pd.read_sql_query("""
WITH booster_keywords AS (
    SELECT DISTINCT cdd.keyword
    FROM adserver_analytics.campaign_daily_data cdd
    WHERE cdd.campaign_type = 'PRODUCT_LISTING'
      AND cdd.sub_campaign_id <> -1
      AND cdd.l1_category_id  = -1
      AND CAST(from_unixtime(cdd.date_ist) AS DATE) >= DATE '2026-04-15'
      AND CAST(insert_ds_ist AS DATE) >= DATE '2026-04-15' - INTERVAL '30' DAY
)
SELECT
    count(*)                                            AS total_booster_keywords,
    count(ckm.keyword)                                  AS keywords_with_ad_l2,
    round(100.0 * count(ckm.keyword) / count(*), 1)     AS pct_covered
FROM booster_keywords b
LEFT JOIN lake_adserver.category_keyword_mapping ckm ON ckm.keyword = b.keyword
""", connection))


## 1a. Keyword → Category dim extract (QC — Loreal shampoo use case)

Per-day extract of `(keyword, ptype, atcs)` for the last 30 days, then local
aggregation to pick the dominant `(l0, l1, l2, ptype)` per keyword via
`dwh.dim_product`. Output ships with the cubes and powers the L0/L1/L2/ptype
dropdowns on trusting-duck.

**Runtime:** ~25-30 min (chunked; resumable via `_DONE` sentinels).
**Coverage:** ~80%+ of booster keywords (limited to keywords that got any ATC).

In [ ]:
# === QC Category Filter: keyword → dominant (L0, L1, L2, ptype) via search_etls ===
#
# Strategy: per-day extract of (keyword, ptype, atcs) from search_etls, then
# local DuckDB aggregation to pick the dominant ptype per keyword. Hoist ptype
# to L0/L1/L2 via a one-shot dim_product pull. Final output:
#   /home/jovyan/final/dim_keyword_category.parquet
# Grain: one row per keyword, columns: keyword, l0, l1, l2, ptype, dominant_share
#
# This feeds the ads cockpit's L0/L1/L2/ptype dropdowns (booster campaigns
# grouped by product category).

# --- SQL templates ---
KEYWORD_CATEGORY_DAILY_SQL = """
WITH cat AS (
  SELECT dp.product_id AS pid, dp.product_type AS ptype
  FROM dwh.dim_product dp
  WHERE dp.is_current = TRUE
    AND dp.l1_Category_id  NOT IN (SELECT DISTINCT l1_category_id FROM category_etls.freebie_categories)
    AND dp.product_type_id NOT IN (SELECT DISTINCT product_type_id FROM category_etls.freebie_categories WHERE product_type_id != 0)
  GROUP BY 1, 2
)
SELECT
  kp.at_date_ist AS day,
  kp.search_query AS keyword,
  c.ptype,
  SUM(kp.atc_qty) AS atcs
FROM search_etls.keyword_city_product_additions kp
INNER JOIN cat c ON c.pid = kp.product_id
WHERE kp.at_date_ist = DATE '{DAY}'
GROUP BY 1, 2, 3
"""

PTYPE_HIERARCHY_SQL = """
SELECT
    product_type                    AS ptype,
    arbitrary(l0_category)          AS l0,
    arbitrary(l1_category)          AS l1,
    arbitrary(l2_category)          AS l2,
    count(DISTINCT product_id)      AS product_count
FROM dwh.dim_product
WHERE is_current = TRUE
GROUP BY 1
"""

# --- Extract: 30-day per-day chunked ---
from datetime import date, timedelta
import os, glob as _glob, sys, site
# duckdb often lands in user site-packages (~/.local/…) which the pencilbox
# kernel excludes by default. Add it explicitly, then install if missing.
_user_site = site.getusersitepackages()
if _user_site not in sys.path:
    sys.path.insert(0, _user_site)
try:
    import duckdb
except ImportError:
    print('installing duckdb via %pip (user site)…')
    get_ipython().run_line_magic('pip', 'install -q --user duckdb')
    import duckdb

KW_CAT_BACKFILL = '/home/jovyan/backfill_kw_category'
os.makedirs(KW_CAT_BACKFILL, exist_ok=True)

CAT_END   = date.today() - timedelta(days=1)
CAT_START = CAT_END - timedelta(days=29)   # 30 days is enough coverage; keeps Trino cheap
print(f'Category window: {CAT_START} → {CAT_END}')

# Weekly batches (checkpoint-resumable, same pattern as booster backfill)
def _weeks(s, e):
    cur = s
    while cur <= e:
        wk_end = min(cur + timedelta(days=6), e)
        yield cur, wk_end
        cur = wk_end + timedelta(days=1)

done_ct = skip_ct = 0
for bs, be in _weeks(CAT_START, CAT_END):
    batch_dir = os.path.join(KW_CAT_BACKFILL, f'{bs}_{be}')
    done_file = os.path.join(batch_dir, '_DONE')
    if os.path.exists(done_file):
        print(f'  ✓ {bs}..{be}: skip (done)')
        skip_ct += 1
        continue
    os.makedirs(batch_dir, exist_ok=True)
    print(f'\n=== Batch {bs} → {be} ===')
    run_to_parquet_chunked(KEYWORD_CATEGORY_DAILY_SQL,
                           os.path.join(batch_dir, 'kw_ptype.parquet'),
                           bs.isoformat(), be.isoformat())
    open(done_file, 'w').close()
    done_ct += 1

print(f'\nkw_category batches done: {done_ct}, skipped: {skip_ct}')

# --- Pull ptype → L0/L1/L2 hierarchy (one-shot) ---
print('\n→ ptype hierarchy (one-shot)')
ptype_hier = sql(PTYPE_HIERARCHY_SQL)
print(f'  {len(ptype_hier):,} ptypes')

# --- Local aggregation: keyword → dominant ptype → l0/l1/l2 ---
print('\n→ aggregating keyword → dominant category')
_con = duckdb.connect(':memory:')
_con.execute("PRAGMA memory_limit='2GB'")

# Write the hierarchy dim to a small parquet for the JOIN
_hier_path = os.path.join(FINAL_DIR, '_ptype_hier.parquet')
ptype_hier.to_parquet(_hier_path, index=False, compression='snappy')

out_path = os.path.join(FINAL_DIR, 'dim_keyword_category.parquet')
_con.execute(f"""
    COPY (
        WITH totals AS (
            SELECT keyword, ptype, SUM(atcs) AS atcs
            FROM read_parquet('{KW_CAT_BACKFILL}/*/kw_ptype.parquet')
            WHERE ptype IS NOT NULL
            GROUP BY 1, 2
        ),
        ranked AS (
            SELECT
                keyword, ptype, atcs,
                atcs * 1.0 / SUM(atcs) OVER (PARTITION BY keyword) AS share,
                ROW_NUMBER() OVER (PARTITION BY keyword ORDER BY atcs DESC) AS rn
            FROM totals
        )
        SELECT
            r.keyword,
            r.ptype,
            h.l0, h.l1, h.l2,
            r.atcs                AS dominant_atcs,
            round(r.share, 4)     AS dominant_share
        FROM ranked r
        LEFT JOIN read_parquet('{_hier_path}') h ON h.ptype = r.ptype
        WHERE rn = 1
    ) TO '{out_path}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

# Sanity
_stats = _con.execute(f"""
    SELECT
        count(*) AS keywords,
        count(l0) AS with_hierarchy,
        avg(dominant_share) AS avg_dominance
    FROM read_parquet('{out_path}')
""").fetchone()
_con.close()

os.remove(_hier_path)
print(f'  {out_path}')
print(f'    keywords: {_stats[0]:,}   with L0/L1/L2: {_stats[1]:,}   avg dominance: {_stats[2]:.1%}')
print(f'    → ships alongside cube_ads on the next ingest_to_booster.sh run')


## 1. SQL templates — three extracts, one source table

Each template uses `{DAY}` (Python `.format`, not f-string) so the day-chunked
runner can substitute it per iteration.

In [ ]:
# ============ Analysis 3 — Booster (keyword-targeted Product Listing) ============
# Per-day extracts carry `campaign_id` only. `campaign_name` is joined in
# from the dim_campaign one-shot in cell 6 → keeps these queries fast.
BOOSTER_KEYWORD_SQL_TEMPLATE = """
SELECT
    CAST(from_unixtime(cdd.date_ist) AS DATE)  AS day,
    trim(a.name)                                AS advertiser,
    cdd.keyword,
    cdd.match_type,
    cdd.campaign_id,
    SUM(impression_served)         AS impressions,
    SUM(clicks)                    AS clicks,
    AVG(most_viewed_position)      AS avg_viewed_position,
    SUM(budget_consumed)           AS budget_consumed,
    SUM(direct_atc)                AS direct_atc,
    SUM(indirect_atc)              AS indirect_atc,
    SUM(direct_sales)              AS direct_sales,
    SUM(indirect_sales)            AS indirect_sales,
    SUM(direct_quantities_sold)    AS direct_qty,
    SUM(indirect_quantities_sold)  AS indirect_qty,
    SUM(new_users_acquired)        AS new_users,
    SUM(campaign_dau)              AS campaign_dau
FROM adserver_analytics.campaign_daily_data cdd
JOIN lake_adserver.campaign   c ON c.id = cdd.campaign_id
JOIN lake_adserver.advertiser a ON a.id = cdd.advertiser_id
WHERE cdd.campaign_type = 'PRODUCT_LISTING'
  AND cdd.sub_campaign_id <> -1
  AND cdd.l1_category_id  = -1
  AND CAST(from_unixtime(cdd.date_ist) AS DATE) = DATE '{DAY}'
  AND CAST(insert_ds_ist AS DATE) >= DATE '{DAY}' - INTERVAL '30' DAY
  AND CAST(insert_ds_ist AS DATE) <= DATE '{DAY}' + INTERVAL '1' DAY
GROUP BY 1, 2, 3, 4, 5
"""

# ============ Analysis 4 — Category Targeting (Product Listing by L1 category) ============
CATEGORY_TARGETING_SQL_TEMPLATE = """
WITH cats AS (
    SELECT id AS l1_category_id, name AS l1_category
    FROM lake_cms.gr_category
    GROUP BY 1, 2
)
SELECT
    CAST(from_unixtime(cdd.date_ist) AS DATE)  AS day,
    trim(a.name)                                AS advertiser,
    cdd.l1_category_id,
    p.l1_category,
    cdd.match_type,
    cdd.campaign_id,
    SUM(impression_served)         AS impressions,
    SUM(clicks)                    AS clicks,
    AVG(most_viewed_position)      AS avg_viewed_position,
    SUM(budget_consumed)           AS budget_consumed,
    SUM(direct_atc)                AS direct_atc,
    SUM(indirect_atc)              AS indirect_atc,
    SUM(direct_sales)              AS direct_sales,
    SUM(indirect_sales)            AS indirect_sales,
    SUM(direct_quantities_sold)    AS direct_qty,
    SUM(indirect_quantities_sold)  AS indirect_qty,
    SUM(new_users_acquired)        AS new_users,
    SUM(campaign_dau)              AS campaign_dau
FROM adserver_analytics.campaign_daily_data cdd
JOIN lake_adserver.campaign   c ON c.id = cdd.campaign_id
JOIN lake_adserver.advertiser a ON a.id = cdd.advertiser_id
LEFT JOIN cats                p ON p.l1_category_id = cdd.l1_category_id
WHERE cdd.campaign_type = 'PRODUCT_LISTING'
  AND cdd.sub_campaign_id <> -1
  AND cdd.l1_category_id <> -1
  AND CAST(from_unixtime(cdd.date_ist) AS DATE) = DATE '{DAY}'
  AND CAST(insert_ds_ist AS DATE) >= DATE '{DAY}' - INTERVAL '30' DAY
  AND CAST(insert_ds_ist AS DATE) <= DATE '{DAY}' + INTERVAL '1' DAY
GROUP BY 1, 2, 3, 4, 5, 6
"""

# ============ Analysis 5 — Reco Assets (Product Recommendation by asset_type) ============
RECO_ASSET_SQL_TEMPLATE = """
SELECT
    CAST(from_unixtime(cdd.date_ist) AS DATE)  AS day,
    trim(a.name)                                AS advertiser,
    CASE
        WHEN sc.targeting_type = 'REPEAT_ORDER_SUGGESTION'        THEN 'Repeat Order Suggestions'
        WHEN sc.targeting_type = 'PRODUCT_SHELF'                  THEN 'Use Case Shelves'
        WHEN sc.targeting_type = 'RECOMMENDATION'                 THEN 'Cart Recommendations'
        WHEN sc.targeting_type = 'NEXT_PRODUCT_RECOMMENDATION'    THEN 'Next Product Recommendations'
        WHEN sc.targeting_type = 'CONTINUE_BROWSING_RECOMMENDATION' THEN 'Continue Browsing'
        ELSE sc.targeting_type
    END                                         AS asset_type,
    cdd.campaign_id,
    SUM(cdd.impression_served)         AS impressions,
    SUM(cdd.clicks)                    AS clicks,
    AVG(cdd.most_viewed_position)      AS avg_viewed_position,
    SUM(cdd.budget_consumed)           AS budget_consumed,
    SUM(cdd.direct_atc)                AS direct_atc,
    SUM(cdd.indirect_atc)              AS indirect_atc,
    SUM(cdd.direct_sales)              AS direct_sales,
    SUM(cdd.indirect_sales)            AS indirect_sales,
    SUM(cdd.direct_quantities_sold)    AS direct_qty,
    SUM(cdd.indirect_quantities_sold)  AS indirect_qty,
    SUM(cdd.new_users_acquired)        AS new_users
FROM adserver_analytics.campaign_daily_data cdd
JOIN lake_adserver.campaign      c  ON c.id  = cdd.campaign_id
JOIN lake_adserver.advertiser    a  ON a.id  = cdd.advertiser_id
JOIN lake_adserver.sub_campaign  sc ON sc.sub_campaign_id = cdd.sub_campaign_id
WHERE cdd.campaign_type = 'PRODUCT_RECOMMENDATION'
  AND cdd.sub_campaign_id <> -1
  AND CAST(from_unixtime(cdd.date_ist) AS DATE) = DATE '{DAY}'
  AND CAST(insert_ds_ist AS DATE) >= DATE '{DAY}' - INTERVAL '30' DAY
  AND CAST(insert_ds_ist AS DATE) <= DATE '{DAY}' + INTERVAL '1' DAY
GROUP BY 1, 2, 3, 4
"""

# ============ NEW: Banner Listing (keyword-targeted, separate campaign_type) ============
# QC 1.3. VERIFY the literal via the probe cell before running the backfill.
CAMPAIGN_TYPE_BANNER = 'BANNER_LISTING'   # ← override if probe reveals different

BANNER_LISTING_SQL_TEMPLATE = f"""
SELECT
    CAST(from_unixtime(cdd.date_ist) AS DATE)  AS day,
    trim(a.name)                                AS advertiser,
    cdd.keyword,
    cdd.match_type,
    cdd.campaign_id,
    SUM(impression_served)         AS impressions,
    SUM(clicks)                    AS clicks,
    AVG(most_viewed_position)      AS avg_viewed_position,
    SUM(budget_consumed)           AS budget_consumed,
    SUM(direct_atc)                AS direct_atc,
    SUM(indirect_atc)              AS indirect_atc,
    SUM(direct_sales)              AS direct_sales,
    SUM(indirect_sales)            AS indirect_sales,
    SUM(direct_quantities_sold)    AS direct_qty,
    SUM(indirect_quantities_sold)  AS indirect_qty,
    SUM(new_users_acquired)        AS new_users
FROM adserver_analytics.campaign_daily_data cdd
JOIN lake_adserver.campaign   c ON c.id = cdd.campaign_id
JOIN lake_adserver.advertiser a ON a.id = cdd.advertiser_id
WHERE cdd.campaign_type = '{CAMPAIGN_TYPE_BANNER}'
  AND cdd.sub_campaign_id <> -1
  AND CAST(from_unixtime(cdd.date_ist) AS DATE) = DATE '{{DAY}}'
  AND CAST(insert_ds_ist AS DATE) >= DATE '{{DAY}}' - INTERVAL '30' DAY
  AND CAST(insert_ds_ist AS DATE) <= DATE '{{DAY}}' + INTERVAL '1' DAY
GROUP BY 1, 2, 3, 4, 5
"""

# ============ NEW: dim_campaign — one-shot pull of campaign_id → name ============
# Runs once in cell 6 (not per-day). Joined into the tier merge + PAN cubes.
# Small (~thousands of rows for any given advertiser × campaign combo).
DIM_CAMPAIGN_SQL = """
WITH agg AS (
    SELECT
        campaign_id,
        SUM(budget_consumed)                                   AS budget_consumed,
        SUM(impression_served)                                 AS impressions,
        MIN(CAST(from_unixtime(date_ist) AS DATE))             AS first_seen,
        MAX(CAST(from_unixtime(date_ist) AS DATE))             AS last_seen,
        MAX(CASE
              WHEN campaign_type = 'BRAND_BOOSTER'  THEN 1
              WHEN l1_category_id <> -1 AND campaign_type = 'PRODUCT_LISTING' THEN 2
              ELSE 0
            END)                                                AS targeting_flag
    FROM adserver_analytics.campaign_daily_data
    WHERE campaign_type IN (
              'PRODUCT_LISTING',
              'BANNER_LISTING',
              'SEARCH_SUGGESTION',
              'PRODUCT_RECOMMENDATION',
              'BRAND_BOOSTER'
          )
      AND sub_campaign_id <> -1
      AND CAST(from_unixtime(date_ist) AS DATE) >= CURRENT_DATE - INTERVAL '180' DAY
      AND CAST(insert_ds_ist        AS DATE) >= CURRENT_DATE - INTERVAL '211' DAY
    GROUP BY 1
)
SELECT
    c.id                                                        AS campaign_id,
    COALESCE(
        JSON_EXTRACT_SCALAR(c.meta_data, '$.campaign_name'),
        trim(c.name)
    )                                                           AS campaign_name,
    a.id                                                        AS advertiser_id,
    trim(a.name)                                                AS advertiser,
    c.campaign_type,
    CASE
        WHEN g.targeting_flag = 1 THEN 'BRAND_BOOSTER'
        WHEN g.targeting_flag = 2 THEN 'Category'
        ELSE c.campaign_type
    END                                                         AS targeting_type,
    COALESCE(g.budget_consumed, 0)                              AS budget_consumed,
    COALESCE(g.impressions,     0)                              AS impressions,
    g.first_seen,
    g.last_seen
FROM lake_adserver.campaign   c
JOIN lake_adserver.advertiser a ON a.id = c.advertiser_id
LEFT JOIN agg                 g ON g.campaign_id = c.id
WHERE c.campaign_type IN (
          'PRODUCT_LISTING',
          'BANNER_LISTING',
          'SEARCH_SUGGESTION',
          'PRODUCT_RECOMMENDATION',
          'BRAND_BOOSTER'
      )
"""

print('SQL templates loaded (per-day extracts w/o campaign_name; dim_campaign standalone, all 5 types + lifetime metrics)')

## 2. Backfill orchestrator — weekly batches, checkpointed

Run this cell to do the 90-day backfill. Each weekly batch lands in
`/home/jovyan/backfill_booster/<week_start>_<week_end>/` with parquets
per analysis + a `_DONE` sentinel. Re-run safely after kernel deaths —
completed weeks skip. `INCLUDE_BANNER_LISTING` controls the QC 1.3 Banner
Listing extract; flip to False if the probe revealed a different literal
than `BANNER_LISTING` (and update cell 3 before re-running).

**Time:** ~3-5 hours for 90 days with all 4 extracts.

In [ ]:
import duckdb
import glob as _glob
import pyarrow.parquet as _pq

# === Incremental refresh (not a full re-extract) ===
# Scans existing batch parquets to find the latest extracted day per source.
# Only extracts from (max_existing_day + 1) forward to yesterday. Progress is
# logged per file so you know it's alive.

BACKFILL_END = date.today() - timedelta(days=1)

# Single duckdb connection reused across all scans (faster than one-shot queries)
_scan_con = duckdb.connect(':memory:')

def _covered_days(source_name):
    """Return the set of ISO-day strings already covered by existing parquets
    for this source. Assumes each parquet covers a contiguous [MIN(day), MAX(day)]
    range — this holds for our per-day chunked extractor. MIN/MAX reads only
    parquet statistics (near-instant per file), so this is much faster than
    SELECT DISTINCT."""
    days = set()
    files = sorted(_glob.glob(f'{BACKFILL_DIR}/*/{source_name}.parquet'))
    print(f'  scanning {len(files)} {source_name} parquets...')
    for i, bf in enumerate(files, 1):
        try:
            schema = _pq.read_schema(bf)
            if 'campaign_id' not in schema.names:
                print(f'    {i}/{len(files)}: OLD schema (no campaign_id) → skip: {os.path.basename(os.path.dirname(bf))}')
                continue
            row = _scan_con.execute(
                f"SELECT MIN(CAST(day AS DATE)), MAX(CAST(day AS DATE)) FROM read_parquet('{bf}')"
            ).fetchone()
            mn, mx = row[0], row[1]
            if mn and mx:
                mn_d = mn if isinstance(mn, date) else date.fromisoformat(str(mn)[:10])
                mx_d = mx if isinstance(mx, date) else date.fromisoformat(str(mx)[:10])
                d = mn_d
                while d <= mx_d:
                    days.add(d.isoformat())
                    d += timedelta(days=1)
                print(f'    {i}/{len(files)}: {os.path.basename(os.path.dirname(bf)):40s} covers {mn_d} → {mx_d}')
            else:
                print(f'    {i}/{len(files)}: {os.path.basename(os.path.dirname(bf))} — empty')
        except Exception as e:
            print(f'    {i}/{len(files)}: WARN could not read {bf}: {e}')
    return days

def _missing_range(source_name):
    covered = _covered_days(source_name)
    want_start = BACKFILL_END - timedelta(days=89)
    d = want_start
    missing = []
    while d <= BACKFILL_END:
        if d.isoformat() not in covered:
            missing.append(d)
        d += timedelta(days=1)
    if not missing:
        return None, None, covered
    return missing[0], missing[-1], covered

# --- One-shot dim: refresh only if > 7 days old OR schema outdated ---
CAMPAIGNS_PATH = os.path.join(FINAL_DIR, 'campaigns.parquet')
CAMPAIGN_DIM_SCHEMA_MIN_COLS = {'campaign_id', 'campaign_name', 'advertiser', 'campaign_type',
                                 'budget_consumed', 'impressions', 'first_seen', 'last_seen',
                                 'targeting_type', 'advertiser_id'}
_stale = True
_schema_outdated = False
if os.path.exists(CAMPAIGNS_PATH):
    _age_days = (time.time() - os.path.getmtime(CAMPAIGNS_PATH)) / 86400
    _stale = _age_days > 7
    try:
        _cols = set(_pq.read_schema(CAMPAIGNS_PATH).names)
        _schema_outdated = not CAMPAIGN_DIM_SCHEMA_MIN_COLS.issubset(_cols)
    except Exception:
        _schema_outdated = True
    reason = 'refreshing (age)' if _stale else ('refreshing (schema)' if _schema_outdated else 'keeping')
    print(f'campaigns.parquet age: {_age_days:.1f} days — {reason}')
if _stale or _schema_outdated:
    print('→ dim_campaign (one-shot refresh, all 5 types + lifetime metrics)')
    dim_campaign = sql(DIM_CAMPAIGN_SQL)
    dim_campaign.to_parquet(CAMPAIGNS_PATH, index=False, compression='snappy')
    print(f'  done {CAMPAIGNS_PATH}: {len(dim_campaign):,} campaigns')

# Also mirror into final/dim/ so ingest_to_pod.sh ships it as
# /var/lib/marketing/dim/dim_campaign.parquet (the pod's campaign filter reads it).
import shutil
_dim_dir = os.path.join(FINAL_DIR, 'dim')
os.makedirs(_dim_dir, exist_ok=True)
_dim_target = os.path.join(_dim_dir, 'dim_campaign.parquet')
shutil.copyfile(CAMPAIGNS_PATH, _dim_target)
print(f'  mirrored → {_dim_target}')

# --- Incremental extract per source ---
INCLUDE_BANNER_LISTING = False

extracted_any = False
for source_name, sql_template in (
    ('booster_keyword',    BOOSTER_KEYWORD_SQL_TEMPLATE),
    ('category_targeting', CATEGORY_TARGETING_SQL_TEMPLATE),
    ('reco_asset',         RECO_ASSET_SQL_TEMPLATE),
    *([('banner_listing', BANNER_LISTING_SQL_TEMPLATE)] if INCLUDE_BANNER_LISTING else []),
):
    start, end, covered = _missing_range(source_name)
    print(f'\n== {source_name}: {len(covered)} days already covered')
    if start is None:
        print(f'  ✓ up to date (through {BACKFILL_END})')
        continue
    print(f'  → extracting {start} → {end} ({(end-start).days + 1} days)')
    inc_dir = os.path.join(BACKFILL_DIR, f'incremental_{start}_{end}')
    os.makedirs(inc_dir, exist_ok=True)
    out_path = os.path.join(inc_dir, f'{source_name}.parquet')
    if os.path.exists(out_path):
        os.remove(out_path)
    run_to_parquet_chunked(sql_template, out_path, start.isoformat(), end.isoformat())
    open(os.path.join(inc_dir, '_DONE'), 'w').close()
    extracted_any = True

_scan_con.close()

if not extracted_any:
    print('\n✓ Nothing new to extract — everything through yesterday is already covered. Skip to cell 8.')
else:
    print('\n✓ Incremental extract complete. Run cell 8 (tier merge) next.')


## 3. Merge → tiered finals for the pod

Same tier policy as the main extracts (hot 14d at day-grain, cold rest at
week-grain, no city dim — ads don't have one). Produces tiered Parquets
in `/home/jovyan/final/`:

- `booster_keyword_hot.parquet` / `_cold.parquet`
- `category_targeting_hot.parquet` / `_cold.parquet`
- `reco_asset_hot.parquet` / `_cold.parquet`
- `banner_listing_hot.parquet` / `_cold.parquet`  *(new)*

All tiered cold files now keep `campaign_id` + `campaign_name` so the
cockpit can offer campaign-level filtering across the 90-day window.

In [ ]:
import duckdb

HOT_CUTOFF = (date.today() - timedelta(days=14)).isoformat()
print(f'Hot cutoff: day >= {HOT_CUTOFF}')

_t0 = time.time()
_con = duckdb.connect(':memory:')

# day column is VARCHAR in the parquet (pandas stringified). Cast at read.
def _cast_day(rel_path):
    return f"(SELECT * REPLACE (CAST(day AS DATE) AS day) FROM read_parquet('{rel_path}'))"

# dim_campaign — small parquet from cell 6 with (campaign_id, campaign_name)
CAMPAIGNS_PATH = os.path.join(FINAL_DIR, 'campaigns.parquet')

# ---- Analysis 3 — Booster keyword ----
_con.execute(f"""
    COPY (
        SELECT s.*, d.campaign_name
        FROM {_cast_day(f'{BACKFILL_DIR}/*/booster_keyword.parquet')} s
        LEFT JOIN read_parquet('{CAMPAIGNS_PATH}') d
          ON d.campaign_id = s.campaign_id
        WHERE s.day >= DATE '{HOT_CUTOFF}'
    )
    TO '{FINAL_DIR}/booster_keyword_hot.parquet' (FORMAT PARQUET, COMPRESSION SNAPPY)
""")
_con.execute(f"""
    COPY (
        SELECT DATE_TRUNC('week', s.day) AS week_start,
               s.advertiser, s.keyword, s.match_type, s.campaign_id,
               COALESCE(d.campaign_name, '(unknown)') AS campaign_name,
               SUM(s.impressions) AS impressions, SUM(s.clicks) AS clicks,
               AVG(s.avg_viewed_position) AS avg_viewed_position,
               SUM(s.budget_consumed) AS budget_consumed,
               SUM(s.direct_atc) AS direct_atc, SUM(s.indirect_atc) AS indirect_atc,
               SUM(s.direct_sales) AS direct_sales, SUM(s.indirect_sales) AS indirect_sales,
               SUM(s.direct_qty) AS direct_qty, SUM(s.indirect_qty) AS indirect_qty,
               SUM(s.new_users) AS new_users
        FROM {_cast_day(f'{BACKFILL_DIR}/*/booster_keyword.parquet')} s
        LEFT JOIN read_parquet('{CAMPAIGNS_PATH}') d
          ON d.campaign_id = s.campaign_id
        WHERE s.day < DATE '{HOT_CUTOFF}'
        GROUP BY 1,2,3,4,5,6
    )
    TO '{FINAL_DIR}/booster_keyword_cold.parquet' (FORMAT PARQUET, COMPRESSION SNAPPY)
""")
print('  booster_keyword tiered')

# ---- Analysis 4 — Category Targeting ----
_con.execute(f"""
    COPY (
        SELECT s.*, d.campaign_name
        FROM {_cast_day(f'{BACKFILL_DIR}/*/category_targeting.parquet')} s
        LEFT JOIN read_parquet('{CAMPAIGNS_PATH}') d ON d.campaign_id = s.campaign_id
        WHERE s.day >= DATE '{HOT_CUTOFF}'
    )
    TO '{FINAL_DIR}/category_targeting_hot.parquet' (FORMAT PARQUET, COMPRESSION SNAPPY)
""")
_con.execute(f"""
    COPY (
        SELECT DATE_TRUNC('week', s.day) AS week_start,
               s.advertiser, s.l1_category_id, s.l1_category, s.match_type, s.campaign_id,
               COALESCE(d.campaign_name, '(unknown)') AS campaign_name,
               SUM(s.impressions) AS impressions, SUM(s.clicks) AS clicks,
               AVG(s.avg_viewed_position) AS avg_viewed_position,
               SUM(s.budget_consumed) AS budget_consumed,
               SUM(s.direct_atc) AS direct_atc, SUM(s.indirect_atc) AS indirect_atc,
               SUM(s.direct_sales) AS direct_sales, SUM(s.indirect_sales) AS indirect_sales,
               SUM(s.direct_qty) AS direct_qty, SUM(s.indirect_qty) AS indirect_qty,
               SUM(s.new_users) AS new_users
        FROM {_cast_day(f'{BACKFILL_DIR}/*/category_targeting.parquet')} s
        LEFT JOIN read_parquet('{CAMPAIGNS_PATH}') d ON d.campaign_id = s.campaign_id
        WHERE s.day < DATE '{HOT_CUTOFF}'
        GROUP BY 1,2,3,4,5,6,7
    )
    TO '{FINAL_DIR}/category_targeting_cold.parquet' (FORMAT PARQUET, COMPRESSION SNAPPY)
""")
print('  category_targeting tiered')

# ---- Analysis 5 — Reco Asset ----
_con.execute(f"""
    COPY (
        SELECT s.*, d.campaign_name
        FROM {_cast_day(f'{BACKFILL_DIR}/*/reco_asset.parquet')} s
        LEFT JOIN read_parquet('{CAMPAIGNS_PATH}') d ON d.campaign_id = s.campaign_id
        WHERE s.day >= DATE '{HOT_CUTOFF}'
    )
    TO '{FINAL_DIR}/reco_asset_hot.parquet' (FORMAT PARQUET, COMPRESSION SNAPPY)
""")
_con.execute(f"""
    COPY (
        SELECT DATE_TRUNC('week', s.day) AS week_start,
               s.advertiser, s.asset_type, s.campaign_id,
               COALESCE(d.campaign_name, '(unknown)') AS campaign_name,
               SUM(s.impressions) AS impressions, SUM(s.clicks) AS clicks,
               AVG(s.avg_viewed_position) AS avg_viewed_position,
               SUM(s.budget_consumed) AS budget_consumed,
               SUM(s.direct_atc) AS direct_atc, SUM(s.indirect_atc) AS indirect_atc,
               SUM(s.direct_sales) AS direct_sales, SUM(s.indirect_sales) AS indirect_sales,
               SUM(s.direct_qty) AS direct_qty, SUM(s.indirect_qty) AS indirect_qty,
               SUM(s.new_users) AS new_users
        FROM {_cast_day(f'{BACKFILL_DIR}/*/reco_asset.parquet')} s
        LEFT JOIN read_parquet('{CAMPAIGNS_PATH}') d ON d.campaign_id = s.campaign_id
        WHERE s.day < DATE '{HOT_CUTOFF}'
        GROUP BY 1,2,3,4,5
    )
    TO '{FINAL_DIR}/reco_asset_cold.parquet' (FORMAT PARQUET, COMPRESSION SNAPPY)
""")
print('  reco_asset tiered')

# ---- Banner Listing (QC 1.3) — only if extract ran ----
import glob as _glob
if _glob.glob(f'{BACKFILL_DIR}/*/banner_listing.parquet'):
    _con.execute(f"""
        COPY (
            SELECT s.*, d.campaign_name
            FROM {_cast_day(f'{BACKFILL_DIR}/*/banner_listing.parquet')} s
            LEFT JOIN read_parquet('{CAMPAIGNS_PATH}') d ON d.campaign_id = s.campaign_id
            WHERE s.day >= DATE '{HOT_CUTOFF}'
        )
        TO '{FINAL_DIR}/banner_listing_hot.parquet' (FORMAT PARQUET, COMPRESSION SNAPPY)
    """)
    _con.execute(f"""
        COPY (
            SELECT DATE_TRUNC('week', s.day) AS week_start,
                   s.advertiser, s.keyword, s.match_type, s.campaign_id,
                   COALESCE(d.campaign_name, '(unknown)') AS campaign_name,
                   SUM(s.impressions) AS impressions, SUM(s.clicks) AS clicks,
                   AVG(s.avg_viewed_position) AS avg_viewed_position,
                   SUM(s.budget_consumed) AS budget_consumed,
                   SUM(s.direct_atc) AS direct_atc, SUM(s.indirect_atc) AS indirect_atc,
                   SUM(s.direct_sales) AS direct_sales, SUM(s.indirect_sales) AS indirect_sales,
                   SUM(s.direct_qty) AS direct_qty, SUM(s.indirect_qty) AS indirect_qty,
                   SUM(s.new_users) AS new_users
            FROM {_cast_day(f'{BACKFILL_DIR}/*/banner_listing.parquet')} s
            LEFT JOIN read_parquet('{CAMPAIGNS_PATH}') d ON d.campaign_id = s.campaign_id
            WHERE s.day < DATE '{HOT_CUTOFF}'
            GROUP BY 1,2,3,4,5,6
        )
        TO '{FINAL_DIR}/banner_listing_cold.parquet' (FORMAT PARQUET, COMPRESSION SNAPPY)
    """)
    print('  banner_listing tiered')

_con.close()

print(f'\ntiered ads finals in {time.time()-_t0:.0f}s:\n{FINAL_DIR}:')
for f in sorted(os.listdir(FINAL_DIR)):
    if any(k in f for k in ('booster','category','reco','banner','campaigns')):
        sz = os.path.getsize(os.path.join(FINAL_DIR, f)) / 1024**2
        print(f'  {f:<35} {sz:>7.1f} MB')

## 4. PAN-cube merge for the pod (Analyses 3 / 4 / 5 / 6 + Banner Listing)

Builds the cubes the pod's `/ads` page reads:

- `cube_booster_keyword.parquet` — (day × advertiser × keyword × campaign_id) + campaign_name
- `cube_category_targeting.parquet` — (day × advertiser × L1_category × campaign_id) + campaign_name
- `cube_reco_asset.parquet` — (day × advertiser × asset_type × campaign_id) + campaign_name
- `cube_banner_listing.parquet` — (day × advertiser × keyword × campaign_id) + campaign_name *(new)*

**What changed (vs the old cube schema):**
- All cubes now carry `campaign_id` + `campaign_name` → cockpit can offer a Campaign filter and a Campaign Performance section (QC 1.1 + 1.5)
- New banner_listing cube → cockpit can split keyword ads into Product Booster vs Banner Listing (QC 1.3)

Output: `/home/jovyan/final/cube_ads/`. Total ~250-400 MB (was ~80-150 MB).
Drop on the pod with `./ingest_to_booster.sh ~/Downloads/cube_ads` after
downloading the directory.

In [ ]:
import duckdb, os, time, glob

ADS_OUT = '/home/jovyan/final/cube_ads'
os.makedirs(ADS_OUT, exist_ok=True)
os.makedirs('/home/jovyan/duckdb_spill', exist_ok=True)

CAMPAIGNS_PATH = os.path.join(FINAL_DIR, 'campaigns.parquet')

_t0 = time.time()

# Batch-by-batch PAN cube build — avoids the 96M-row single GROUP BY that OOMs
# even with disk spill. Each batch is a week (~1-2M rows), fits comfortably in
# memory, and the union at the end is cheap because per-batch aggregation
# already collapsed rows.
def build_cube_two_pass(pattern, group_cols, agg_sql, out_file, needs_campaign_join=True):
    con = duckdb.connect(':memory:')
    con.execute("PRAGMA memory_limit='3GB'")
    con.execute("PRAGMA temp_directory='/home/jovyan/duckdb_spill'")
    con.execute("PRAGMA threads=2")
    tmp_dir = f'/home/jovyan/duckdb_spill/_pass1_{os.path.basename(out_file)}'
    os.makedirs(tmp_dir, exist_ok=True)
    for f in os.listdir(tmp_dir):
        os.remove(os.path.join(tmp_dir, f))
    batch_files = sorted(glob.glob(pattern))
    print(f'  pass 1 · aggregating {len(batch_files)} batches → {tmp_dir}/')
    join_clause = ""
    if needs_campaign_join:
        join_clause = f"LEFT JOIN read_parquet('{CAMPAIGNS_PATH}') d ON d.campaign_id = s.campaign_id"
    for i, bf in enumerate(batch_files):
        tmp_out = os.path.join(tmp_dir, f'batch_{i:03d}.parquet')
        con.execute(f"""
            COPY (
                SELECT {group_cols},
                       {agg_sql}
                FROM (SELECT * REPLACE (CAST(day AS DATE) AS day) FROM read_parquet('{bf}')) s
                {join_clause}
                GROUP BY ALL
            ) TO '{tmp_out}' (FORMAT PARQUET, COMPRESSION SNAPPY)
        """)
    # Pass 2 — union pass-1 outputs + final aggregate. Rows already collapsed,
    # so this is cheap. ANY_VALUE → MAX so the outer aggregate still legal.
    print(f'  pass 2 · union {len(batch_files)} tmp files → final cube')
    import re as _re
    # Strip s. and d. prefixes for pass 2 — tmp files have already-aggregated
    # unqualified column names.
    agg_sql_pass2 = _re.sub(r'\b[sd]\.', '', agg_sql).replace('ANY_VALUE', 'MAX')
    con.execute(f"""
        COPY (
            SELECT {group_cols.replace('s.', '')},
                   {agg_sql_pass2}
            FROM read_parquet('{tmp_dir}/*.parquet') s
            GROUP BY ALL
        ) TO '{out_file}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    con.close()
    for f in os.listdir(tmp_dir):
        os.remove(os.path.join(tmp_dir, f))
    os.rmdir(tmp_dir)

# ---- Booster keyword cube (Analysis 3 source, QC 1.1 + 1.5) ----
build_cube_two_pass(
    pattern=f'{BACKFILL_DIR}/*/booster_keyword.parquet',
    group_cols='s.day, s.advertiser, s.keyword, s.campaign_id',
    agg_sql="""ANY_VALUE(d.campaign_name) AS campaign_name,
               SUM(s.impressions)         AS impressions,
               SUM(s.clicks)              AS clicks,
               AVG(s.avg_viewed_position) AS avg_viewed_position,
               SUM(s.budget_consumed)     AS budget_consumed,
               SUM(s.direct_atc)          AS direct_atc,
               SUM(s.indirect_atc)        AS indirect_atc,
               SUM(s.direct_sales)        AS direct_sales,
               SUM(s.indirect_sales)      AS indirect_sales,
               SUM(s.direct_qty)          AS direct_qty,
               SUM(s.indirect_qty)        AS indirect_qty,
               SUM(s.new_users)           AS new_users,
               SUM(s.campaign_dau)        AS campaign_dau""",
    out_file=f'{ADS_OUT}/cube_booster_keyword.parquet',
)
print(f'  cube_booster_keyword ({time.time()-_t0:.0f}s)')

# ---- Category targeting cube ----
build_cube_two_pass(
    pattern=f'{BACKFILL_DIR}/*/category_targeting.parquet',
    group_cols='s.day, s.advertiser, s.l1_category_id, s.campaign_id',
    agg_sql="""ANY_VALUE(s.l1_category)    AS l1_category,
               ANY_VALUE(d.campaign_name) AS campaign_name,
               SUM(s.impressions)         AS impressions,
               SUM(s.clicks)              AS clicks,
               AVG(s.avg_viewed_position) AS avg_viewed_position,
               SUM(s.budget_consumed)     AS budget_consumed,
               SUM(s.direct_atc)          AS direct_atc,
               SUM(s.indirect_atc)        AS indirect_atc,
               SUM(s.direct_sales)        AS direct_sales,
               SUM(s.indirect_sales)      AS indirect_sales,
               SUM(s.direct_qty)          AS direct_qty,
               SUM(s.indirect_qty)        AS indirect_qty,
               SUM(s.new_users)           AS new_users,
               SUM(s.campaign_dau)        AS campaign_dau""",
    out_file=f'{ADS_OUT}/cube_category_targeting.parquet',
)
print(f'  cube_category_targeting ({time.time()-_t0:.0f}s)')

# ---- Reco asset cube ----
build_cube_two_pass(
    pattern=f'{BACKFILL_DIR}/*/reco_asset.parquet',
    group_cols='s.day, s.advertiser, s.asset_type, s.campaign_id',
    agg_sql="""ANY_VALUE(d.campaign_name) AS campaign_name,
               SUM(s.impressions)         AS impressions,
               SUM(s.clicks)              AS clicks,
               AVG(s.avg_viewed_position) AS avg_viewed_position,
               SUM(s.budget_consumed)     AS budget_consumed,
               SUM(s.direct_atc)          AS direct_atc,
               SUM(s.indirect_atc)        AS indirect_atc,
               SUM(s.direct_sales)        AS direct_sales,
               SUM(s.indirect_sales)      AS indirect_sales,
               SUM(s.direct_qty)          AS direct_qty,
               SUM(s.indirect_qty)        AS indirect_qty,
               SUM(s.new_users)           AS new_users""",
    out_file=f'{ADS_OUT}/cube_reco_asset.parquet',
)
print(f'  cube_reco_asset ({time.time()-_t0:.0f}s)')

# ---- Banner Listing cube (QC 1.3) — only if extract ran ----
if glob.glob(f'{BACKFILL_DIR}/*/banner_listing.parquet'):
    build_cube_two_pass(
        pattern=f'{BACKFILL_DIR}/*/banner_listing.parquet',
        group_cols='s.day, s.advertiser, s.keyword, s.campaign_id',
        agg_sql="""ANY_VALUE(d.campaign_name) AS campaign_name,
                   SUM(s.impressions)         AS impressions,
                   SUM(s.clicks)              AS clicks,
                   AVG(s.avg_viewed_position) AS avg_viewed_position,
                   SUM(s.budget_consumed)     AS budget_consumed,
                   SUM(s.direct_atc)          AS direct_atc,
                   SUM(s.indirect_atc)        AS indirect_atc,
                   SUM(s.direct_sales)        AS direct_sales,
                   SUM(s.indirect_sales)      AS indirect_sales,
                   SUM(s.direct_qty)          AS direct_qty,
                   SUM(s.indirect_qty)        AS indirect_qty,
                   SUM(s.new_users)           AS new_users""",
        out_file=f'{ADS_OUT}/cube_banner_listing.parquet',
    )
    print(f'  cube_banner_listing ({time.time()-_t0:.0f}s)')

def _sz(p): return os.path.getsize(p)/1024**2
print(f'\n{ADS_OUT}:')
for f in sorted(os.listdir(ADS_OUT)):
    print(f'  {f:<35} {_sz(os.path.join(ADS_OUT, f)):>7.1f} MB')
print(f'\n→ download `{ADS_OUT}` to laptop, ship to pod with ingest_to_booster.sh')

## Done

Six Parquets land in `/home/jovyan/final/` (3 analyses × 2 tiers). When the
pod's Analysis 3 / 4 / 5 / 6 modules ship, `ingest_to_pod.sh` will pick
them up by name.